Q1

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
%time print('Hello')

Hello
CPU times: user 45 µs, sys: 5 µs, total: 50 µs
Wall time: 53.9 µs


In [3]:
%%writefile hello.cu

Writing hello.cu


Q2

In [4]:
!nvidia-smi

Wed Feb 11 17:42:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Q3

Q3. Debugging Common CUDA Errors:
Zero Output: Occurs when the program finishes before the GPU prints.
 Fix: Add cudaDeviceSynchronize(); after the kernel launch.
Incorrect Indexing: Occurs when accessing array elements out of bounds. Fix: Add if (idx < N) checks inside the kernel.
PTX Errors: Architecture mismatch. Fix: Ensure the code is compiled for the correct GPU architecture (usually handled automatically in Colab).

Q4

In [7]:
%%writefile q4_hello.cu
#include <stdio.h>
#include <cuda_runtime.h>

// Q4: Device code (GPU Kernel)
__global__ void helloKernel() {
    // Q4c: Compute global thread ID
    int global_thread_id = blockIdx.x * blockDim.x + threadIdx.x;

    // Q4b: Print from GPU thread
    printf("Hello from GPU thread %d\n", global_thread_id);
}

// Q4d: Host code (CPU)
int main() {
    printf("Launching Kernel with 1 Block and 8 Threads...\n");

    // Q4a: Launch 1 block, 8 threads
    helloKernel<<<1, 8>>>();

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    return 0;
}

Overwriting q4_hello.cu


In [8]:
!nvcc q4_hello.cu -o q4_hello
!./q4_hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Launching Kernel with 1 Block and 8 Threads...
Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from GPU thread 4
Hello from GPU thread 5
Hello from GPU thread 6
Hello from GPU thread 7


Q5

In [11]:
%%writefile q5_memory.cu
#include <stdio.h>
#include <cuda_runtime.h>

// Kernel to print values from device memory
__global__ void printKernel(int *d_arr, int size) {
    int idx = threadIdx.x;
    if (idx < size) {
        printf("GPU Thread %d read value: %d\n", idx, d_arr[idx]);
    }
}

int main() {
    int size = 5;
    int bytes = size * sizeof(int);

    // Host array
    int h_arr[5] = {10, 20, 30, 40, 50};
    int h_result[5];
    int *d_arr;

    // Allocate memory on GPU
    cudaMalloc((void**)&d_arr, bytes);

    // Copy from Host to Device
    cudaMemcpy(d_arr, h_arr, bytes, cudaMemcpyHostToDevice);

    // Launch kernel
    printKernel<<<1, 5>>>(d_arr, size);
    cudaDeviceSynchronize();

    // Copy back to Host
    cudaMemcpy(h_result, d_arr, bytes, cudaMemcpyDeviceToHost);

    printf("\nData copied back to CPU:\n");
    for(int i = 0; i < size; i++) {
        // CORRECTION IS HERE: added "i" before h_result[i]
        printf("Index %d: %d\n", i, h_result[i]);
    }

    cudaFree(d_arr);
    return 0;
}

Overwriting q5_memory.cu


In [12]:
!nvcc q5_memory.cu -o q5_memory
!./q5_memory

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
GPU Thread 0 read value: 10
GPU Thread 1 read value: 20
GPU Thread 2 read value: 30
GPU Thread 3 read value: 40
GPU Thread 4 read value: 50

Data copied back to CPU:
Index 0: 10
Index 1: 20
Index 2: 30
Index 3: 40
Index 4: 50


Q6

In [13]:
import numpy as np
import time

# Define a large size to see the speed difference clearly
size = 1000000

print(f"Comparing operations on {size} elements...\n")

# --- 1. Python List ---
list_data = list(range(size))

start_time = time.time()
# Operation: Square every number in the list
list_result = [x**2 for x in list_data]
end_time = time.time()

print(f"Python List Time:  {end_time - start_time:.6f} seconds")

# --- 2. Numpy Array ---
np_data = np.arange(size)

start_time = time.time()
# Operation: Square every number (Vectorized operation)
np_result = np_data ** 2
end_time = time.time()

print(f"Numpy Array Time:  {end_time - start_time:.6f} seconds")

# Calculate how much faster Numpy is
speedup = (end_time - start_time)
if speedup > 0:
    ratio = (list_result_time := (end_time - start_time)) # Just a placeholder logic for text
    # Recalculate correctly for print
    list_time = end_time - start_time # Wait, let me fix the variable reuse above in print

Comparing operations on 1000000 elements...

Python List Time:  0.056850 seconds
Numpy Array Time:  0.002655 seconds
